# Module 1 · Foundations of Causal Inference & the DoWhy Workflow
**⏱ 40 minutes** · 0:10 – 0:50

**Learning objectives** — by the end of this module you can:
1. Define potential outcomes, ITE / ATE / ATT / CATE, and the assumptions that make them estimable.
2. Read a causal DAG: forks (confounders), chains (mediators), colliders — and know which to adjust for.
3. Run DoWhy's **Model → Identify → Estimate → Refute** workflow end to end.
4. Spot two classic traps: **bad controls** and **collider (selection) bias**.

**Scenario A:** Does granting a **credit line increase (CLI)** *cause* higher 6-month card spend — and by how much?

In [ ]:
import sys, pathlib, warnings, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings("ignore")
for name in ["dowhy", "dowhy.causal_model", "dowhy.causal_estimator", "dowhy.causal_refuters"]:
    logging.getLogger(name).setLevel(logging.ERROR)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from dowhy import CausalModel
from causal_utils import make_credit_line_data, CREDIT_LINE_EDGES, CREDIT_LINE_POS, edges_to_dot, plot_dag

pd.set_option("display.precision", 2)
plt.rcParams["figure.dpi"] = 100
df = make_credit_line_data(n=10_000, seed=42)
TRUE_ATE = df["true_cate"].mean()
TRUE_ATT = df.loc[df.cli == 1, "true_cate"].mean()
print(f"True ATE = ${TRUE_ATE:,.0f}   True ATT = ${TRUE_ATT:,.0f}")
df.head()

## 1.1 Potential outcomes (Neyman–Rubin)

For each customer $i$ imagine two potential outcomes:
- $Y_i(1)$ — spend **if** granted a CLI
- $Y_i(0)$ — spend **if not**

| Quantity | Definition | Business meaning |
|---|---|---|
| ITE | $\tau_i = Y_i(1) - Y_i(0)$ | Effect for one customer (never observed!) |
| **ATE** | $E[Y(1) - Y(0)]$ | Effect if we gave *everyone* a CLI |
| **ATT** | $E[Y(1) - Y(0) \mid T=1]$ | Effect on those we actually treated |
| **CATE** | $E[Y(1) - Y(0) \mid X=x]$ | Effect for a segment with features $x$ → Module 2 |

**The fundamental problem of causal inference:** we only ever observe *one* of $Y_i(1), Y_i(0)$. Causal inference is a missing-data problem.

### Identification assumptions
1. **Consistency / SUTVA** – one version of treatment; my CLI doesn't change your spend.
2. **Conditional exchangeability (no unmeasured confounding)** – $Y(1), Y(0) \perp T \mid X$.
3. **Positivity / overlap** – $0 < P(T=1 \mid X) < 1$ for all relevant $X$.

Assumption 2 is *untestable* from data alone — which is why we write it down as a graph and stress-test it.

In [ ]:
# Because we simulated the data, we can "peek" at the counterfactual world.
# In real life the column `true_cate` does not exist.
peek = df[["income_k", "credit_score", "cli", "spend", "true_cate"]].head(6).copy()
peek["Y(1)"] = np.where(peek.cli == 1, peek.spend, peek.spend + peek.true_cate)
peek["Y(0)"] = np.where(peek.cli == 0, peek.spend, peek.spend - peek.true_cate)
peek.loc[peek.cli == 1, "Y(0)"] = np.nan  # what we'd actually see...
peek.loc[peek.cli == 0, "Y(1)"] = np.nan
peek  # ← every row is missing one potential outcome

## 1.2 Causal graphs (DAGs): the three building blocks

| Structure | Shape | Example | Adjust for the middle node? |
|---|---|---|---|
| **Fork** (confounder) | $T \leftarrow Z \rightarrow Y$ | income → CLI, income → spend | ✅ **Yes** – blocks the backdoor path |
| **Chain** (mediator) | $T \rightarrow M \rightarrow Y$ | CLI → new balance → spend | ❌ **No** (for the *total* effect) |
| **Collider** | $T \rightarrow C \leftarrow Y$ | score → approved ← income | ❌ **Never** – opens a spurious path |

**Backdoor criterion:** a set $Z$ is a valid adjustment set if it (i) blocks every path from $T$ to $Y$ that starts with an arrow *into* $T$, and (ii) contains no descendant of $T$.

### Quick demo: collider bias (Berkson's paradox) in lending
Suppose income and credit score were *independent* in the population, and loans are approved when a combined score is high. Analysts who only look at **approved** loans will "discover" a negative relationship that doesn't exist.

In [ ]:
rng = np.random.default_rng(0)
inc = rng.normal(0, 1, 5000)
score = rng.normal(0, 1, 5000)                 # independent of income by construction
approved = (inc + score + rng.normal(0, 0.5, 5000)) > 1.0   # collider

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
ax[0].scatter(inc, score, s=3, alpha=.3); ax[0].set_title(f"All applicants  corr={np.corrcoef(inc, score)[0,1]:+.2f}")
ax[1].scatter(inc[approved], score[approved], s=3, alpha=.3, color="C3")
ax[1].set_title(f"Approved only  corr={np.corrcoef(inc[approved], score[approved])[0,1]:+.2f}")
for a in ax: a.set_xlabel("income (std)"); ax[0].set_ylabel("credit score (std)")
plt.tight_layout(); plt.show()

**Lesson:** conditioning on a collider (here: *selecting* on approval) manufactures correlation. Watch for this whenever a dataset is filtered by a decision — approved loans, investigated alerts, retained customers.

---
## 1.3 The DoWhy workflow

```
   ① MODEL        ② IDENTIFY          ③ ESTIMATE            ④ REFUTE
 causal graph → estimand (math) → statistical estimate → robustness checks
 (assumptions)   (backdoor / IV /     (regression, PSM,     (placebo, random cause,
                  frontdoor)           IPW, DML ...)         subset, sensitivity)
```

DoWhy's philosophy: **separate identification (causal assumptions) from estimation (statistics)** and make every assumption explicit.

### ① Model — encode domain knowledge as a DAG

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_dag(CREDIT_LINE_EDGES, ax=ax, pos=CREDIT_LINE_POS,
         title="Assumed causal graph – Scenario A (credit line increase)")
plt.show()

Reading the graph:
- `income_k, credit_score, tenure, utilization, age` → **confounders** (affect both CLI decision and spend)
- `promo_email` → randomized nudge that affects **only** CLI uptake → a candidate **instrument**
- `new_balance` → **mediator** (caused by the CLI) → must *not* be adjusted for
- `region_code` is absent from the graph → we assume it is irrelevant

In [ ]:
model_cols = ["age", "income_k", "credit_score", "tenure", "utilization",
              "promo_email", "cli", "new_balance", "spend"]

model = CausalModel(
    data=df[model_cols],
    treatment="cli",
    outcome="spend",
    graph=edges_to_dot(CREDIT_LINE_EDGES),
)
print("Treatment:", model._treatment, "| Outcome:", model._outcome)

### ② Identify — let DoWhy derive the estimand from the graph
DoWhy applies do-calculus rules (backdoor, instrumental variable, frontdoor) and tells you *which* adjustment set to use. Notice it **automatically excludes** `new_balance` (a descendant of treatment).

In [ ]:
identified = model.identify_effect(proceed_when_unidentifiable=True)
print(identified)

### ③ Estimate — many estimators, one estimand
The same backdoor estimand can be estimated many ways. Let's compare them against the truth.

| Method | Idea | Watch out for |
|---|---|---|
| Linear regression | Model $E[Y \mid T, X]$ | Misspecification; with heterogeneity, plain OLS ≠ ATE |
| Propensity score matching | Compare similar-propensity treated/untreated | Discards data, variance |
| Inverse propensity weighting | Reweight to mimic an RCT | Extreme weights when overlap is poor |
| Stratification | Compare within propensity bins | Residual confounding within bins |
| Instrumental variable | Use a nudge that only moves $T$ | Estimates a **LATE** (compliers); needs a strong, valid instrument |
| Double ML (via EconML) | ML nuisance models + orthogonalization | Next module! |

In [ ]:
estimators = {
    "Linear regression (+ effect modifiers)": dict(method_name="backdoor.linear_regression",
                                                    effect_modifiers=["income_k", "credit_score", "utilization"]),
    "Propensity score matching": dict(method_name="backdoor.propensity_score_matching"),
    "Inverse propensity weighting": dict(method_name="backdoor.propensity_score_weighting"),
    "Propensity score stratification": dict(method_name="backdoor.propensity_score_stratification"),
    "Instrumental variable (promo_email)": dict(method_name="iv.instrumental_variable"),
}
import copy
results, estimates = {}, {}
for label, kw in estimators.items():
    # deepcopy: DoWhy annotates the estimand object in place (e.g. marks it "iv")
    est = model.estimate_effect(copy.deepcopy(identified), target_units="ate", **kw)
    estimates[label] = est
    results[label] = est.value

naive = df.loc[df.cli == 1, "spend"].mean() - df.loc[df.cli == 0, "spend"].mean()
res = pd.Series({"Naive difference in means": naive, **results}).to_frame("estimate ($)")
res["error vs true ATE ($)"] = res["estimate ($)"] - TRUE_ATE
res

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
colors = ["#c0392b"] + ["#2f5d8a"] * (len(res) - 1)
ax.barh(res.index[::-1], res["estimate ($)"][::-1], color=colors[::-1])
ax.axvline(TRUE_ATE, color="black", ls="--", label=f"True ATE ${TRUE_ATE:,.0f}")
ax.set_xlabel("Estimated effect of CLI on 6-month spend ($)"); ax.legend()
plt.tight_layout(); plt.show()

**Discussion prompts**
- Every adjusted estimator lands far closer to the truth than the naive one. *Assumptions* did the heavy lifting, not fancy algorithms.
- IPW tends to be noisier — why? (hint: look at the overlap plot below)
- The IV estimate is a **Local** ATE — the effect for customers whose CLI uptake was changed by the email ("compliers").

In [ ]:
# Positivity / overlap diagnostic: estimated propensity scores by group
from sklearn.linear_model import LogisticRegression
Xc = df[["age", "income_k", "credit_score", "tenure", "utilization"]]
ps = LogisticRegression(max_iter=2000).fit(Xc, df.cli).predict_proba(Xc)[:, 1]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(ps[df.cli == 1], bins=50, alpha=.6, label="CLI granted", density=True)
ax.hist(ps[df.cli == 0], bins=50, alpha=.6, label="No CLI", density=True)
ax.set_xlabel("estimated propensity P(CLI | X)"); ax.legend(); ax.set_title("Overlap check")
plt.tight_layout(); plt.show()
print(f"Share of customers with propensity < 0.05 or > 0.95: {np.mean((ps<.05)|(ps>.95)):.1%}  → large IPW weights")

### ④ Refute — try to break your own estimate
A refuter changes the problem in a way where we **know** what should happen, and checks the estimate behaves.

| Refuter | What it does | Expected if estimate is sound |
|---|---|---|
| Placebo treatment | Replace $T$ with a random permutation | Effect → **≈ 0** |
| Random common cause | Add an independent random confounder | Effect **unchanged** |
| Data subset | Re-estimate on random 80% subsets | Effect **stable** |
| Unobserved common cause | Simulate a hidden confounder of given strength | Shows **how fragile** the estimate is |

In [ ]:
base = estimates["Linear regression (+ effect modifiers)"]   # each estimate carries its own estimand
refutations = [
    ("placebo_treatment_refuter", dict(placebo_type="permute", num_simulations=10)),
    ("random_common_cause", dict(num_simulations=10)),
    ("data_subset_refuter", dict(subset_fraction=0.8, num_simulations=10)),
]
for method, kw in refutations:
    r = model.refute_estimate(base.target_estimand, base, method_name=method, random_seed=1, **kw)
    print(r, "\n")

#### Sensitivity analysis: *how strong would a hidden confounder need to be?*
No data can prove there is no unmeasured confounder. Instead we ask: **how strong would it have to be to change our decision?** Below we sweep the confounder's strength on treatment (probability of flipping CLI) and on outcome ($ effect on spend).

In [ ]:
sens = model.refute_estimate(
    base.target_estimand, base, method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="binary_flip",
    confounders_effect_on_outcome="linear",
    effect_strength_on_treatment=[0.02, 0.05, 0.10, 0.20],
    effect_strength_on_outcome=[200, 500, 1000, 2000],
    random_seed=1,
)
plt.show()
print(sens)

**Reading the heatmap:** each cell is the re-estimated effect under a simulated hidden confounder of that strength. Weak confounders barely move the estimate. Even a confounder that flips ~20% of CLI decisions **and** moves spend by \$1,000+ only pulls it down to roughly \$100–200 — still clearly positive. So the *direction* ("CLIs increase spend") is robust; the exact *magnitude* is more fragile.

**How to communicate this to stakeholders:** *"Our estimate stays positive unless an unmeasured factor both flips ≥ X% of CLI decisions **and** shifts spend by ≥ $Y. Is there a plausible variable that strong we haven't measured?"* — this turns an untestable assumption into a concrete business discussion.

---
## 1.4 The bad-control trap
A well-meaning analyst says *"let's control for everything we have, including the customer's balance after the increase."* Watch what happens:

In [ ]:
confounders = "age + income_k + credit_score + tenure + utilization"
good = smf.ols(f"spend ~ cli + {confounders}", data=df).fit().params["cli"]
bad = smf.ols(f"spend ~ cli + {confounders} + new_balance", data=df).fit().params["cli"]
print(f"Adjust for confounders only          : ${good:,.0f}")
print(f"... AND the mediator new_balance      : ${bad:,.0f}   ← wrong question answered")
print(f"True total effect (ATE)              : ${TRUE_ATE:,.0f}")

Adjusting for the mediator blocks the path **CLI → new_balance → spend**, so you estimate only the *direct* effect (~$300 smaller by construction). **"More controls" is not always safer** — post-treatment variables are poison for total-effect estimation. The graph tells you which variables to include.

---
## ✋ Exercise 1.1 (5 min) — the cost of a wrong graph
Your colleague believes `new_balance` is a **confounder** (i.e. `new_balance → cli` and `new_balance → spend`) rather than a mediator.
1. Build a DoWhy `CausalModel` using that (wrong) graph: copy `CREDIT_LINE_EDGES`, remove `("cli", "new_balance")`, add `("new_balance", "cli")`.
2. Identify and estimate with `backdoor.linear_regression`.
3. Compare to the truth. Would a placebo refuter catch this mistake? (Try it!)

In [ ]:
# YOUR CODE HERE
wrong_edges = ...

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
wrong_edges = [e for e in CREDIT_LINE_EDGES if e != ("cli", "new_balance")] + [("new_balance", "cli")]
wrong_model = CausalModel(data=df[model_cols], treatment="cli", outcome="spend",
                          graph=edges_to_dot(wrong_edges))
wrong_id = wrong_model.identify_effect(proceed_when_unidentifiable=True)
wrong_est = wrong_model.estimate_effect(wrong_id, method_name="backdoor.linear_regression")
print(f"Wrong-graph estimate: ${wrong_est.value:,.0f} vs truth ${TRUE_ATE:,.0f}")
print(wrong_model.refute_estimate(wrong_id, wrong_est, method_name="placebo_treatment_refuter",
                                  placebo_type="permute", num_simulations=5, random_seed=1))
# Takeaway: the placebo refuter PASSES (≈0) even though the estimate is badly biased.
# Refuters test the estimator given the graph; they cannot validate the graph itself.
# Only domain knowledge (timing: balance is measured AFTER the CLI) settles it.
```
</details>

## ✋ Exercise 1.2 (3 min) — ATT vs ATE
The business only cares about the customers who *actually* received a CLI. Re-estimate with `target_units="att"` using `backdoor.propensity_score_matching` and compare with `TRUE_ATT`. Why do ATE and ATT differ here?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>👀 Click to reveal a solution</b> (try it yourself first!)</summary>

```python
att = model.estimate_effect(identified, method_name="backdoor.propensity_score_matching", target_units="att")
print(f"Estimated ATT ${att.value:,.0f} | True ATT ${TRUE_ATT:,.0f} | True ATE ${TRUE_ATE:,.0f}")
# ATT > ATE because the bank preferentially grants CLIs to high-income / high-score
# customers -- exactly the segment with the largest true effect (heterogeneity + selection).
```
</details>

## 🔑 Key takeaways — Module 1
1. **Causal effects are about counterfactuals**; we never observe both potential outcomes.
2. **The DAG is the model.** It decides what to adjust for — confounders yes, mediators and colliders no.
3. **DoWhy** forces the discipline: *Model → Identify → Estimate → Refute*.
4. Refuters catch estimator problems, **not graph problems**; sensitivity analysis quantifies how wrong the "no hidden confounding" assumption could be before it matters.
5. ATE, ATT and LATE (IV) answer **different questions** — pick the one your decision needs.

☕ **Break (10 min)** → then `02_Heterogeneous_Effects_EconML.ipynb`